# PLEASE USE Kaggle GPU T4 (x2) as here both of them are used simultaneously for faster inference

## 1. Dependency Installation and Environment Setup


# Please restart after installations to avoid import errors

In [ ]:
%%capture
# ── Install all required packages ──
# transformers + accelerate for HuggingFace Whisper inference
# silero-vad for VAD segmentation, bnunicodenormalizer for Bengali text normalization
# datasets for efficient batched GPU inference via Dataset + KeyDataset
!pip install -q transformers accelerate silero-vad bnunicodenormalizer jiwer librosa soundfile torchaudio huggingface_hub datasets

# 2. Imports

In [ ]:
# ── Core Imports ──
import os
import gc
import re
import sys
import glob
import time
import logging
import warnings
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import librosa
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from transformers import pipeline as hf_pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from bnunicodenormalizer import Normalizer

# ── Logging Configuration ──
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("BengaliASR")

warnings.filterwarnings("ignore")
print("All imports successful. Environment ready.")

## 3. Directories

- **Silero VAD**: Cached via `torch.hub` to a local directory


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL & DATA PATHS
# ══════════════════════════════════════════════════════════════════════════════

# Whisper model — downloaded from HuggingFace at runtime
# Using mozilla-ai/whisper-large-v3-bn (Bengali fine-tuned) for best Bengali quality
WHISPER_MODEL_ID = "zarifmahir21/whisper-medium-bangla"

# Competition data directories
TEST_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/audio"
TRAIN_ANNOTATION_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/annotation"
SAMPLE_SUBMISSION_PATH = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/sample_submission .csv"

# Submission output path
SUBMISSION_PATH = "./submission.csv"

# Silero VAD local cache directory
VAD_CACHE_DIR = "./silero_vad_cache"

# ══════════════════════════════════════════════════════════════════════════════
# TRANSCRIPTION HYPERPARAMETERS (tuned for Bengali long-form)
# ══════════════════════════════════════════════════════════════════════════════
LANGUAGE = "bn"
BEAM_SIZE = 4
BEST_OF = 4
TEMPERATURE = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)  # Fallback temperatures
REPETITION_PENALTY = 1.1
CONDITION_ON_PREVIOUS_TEXT = False    # Prevent hallucination propagation
COMPRESSION_RATIO_THRESHOLD = 2.4    # Detect repetitive hallucinations
LOG_PROB_THRESHOLD = -1.0
NO_SPEECH_THRESHOLD = 0.6
BATCH_SIZE = 13                  # GPU batch size for batched pipeline inference

# ── Download and cache Silero VAD ──
os.makedirs(VAD_CACHE_DIR, exist_ok=True)
os.environ["TORCH_HOME"] = VAD_CACHE_DIR

print("Loading Silero VAD model...")
vad_model, vad_utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    trust_repo=True,
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils
print("Silero VAD model loaded and cached successfully.")

print(f"Whisper model: {WHISPER_MODEL_ID} (will download from HuggingFace)")
print(f"Beam={BEAM_SIZE}, RepPenalty={REPETITION_PENALTY}, BatchSize={BATCH_SIZE}")
print(f"Temperatures={TEMPERATURE}")

## 3. GPU Configuration and RTF Optimizations

In [ ]:
# ── GPU & cuDNN Optimizations ──
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} | Memory: {gpu_mem:.1f} GB")
    print("cuDNN benchmark enabled for optimized kernel selection.")
else:
    logger.warning("No CUDA GPU detected. Pipeline will be slow on CPU.")


def clear_cuda_cache():
    """Flush GPU memory to prevent OOM on long files."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


print("GPU configuration complete.")

# 4. Dual T4 GPU — Threading-Based Parallel Inference


In [ ]:
import torch

NUM_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GB")

USE_DUAL_GPU = NUM_GPUS >= 2
print(f"\nDual-GPU mode: {'ENABLED' if USE_DUAL_GPU else 'DISABLED — single-GPU fallback'}")


## 4. Audio Loading and Signal Processing Utility

In [ ]:
import torchaudio
import torchaudio.transforms as T

TARGET_SR = 16000

def load_audio(filepath: str) -> np.ndarray:
    """Fast audio loading using PyTorch C++ backend"""
    try:
        # Load audio instantly
        waveform, sr = torchaudio.load(filepath)

        # Resample only if necessary
        if sr != TARGET_SR:
            resampler = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform = resampler(waveform)

        # Convert stereo to mono if necessary
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        return waveform.squeeze().numpy()
    except Exception as e:
        logger.error(f"Failed to load audio {filepath}: {e}")
        return np.array([], dtype=np.float32)

## 5. Silero VAD — Tuned for Bengali Speech

Silero VAD identifies speech regions with a configurable probability threshold.
Parameters tuned for Bengali: lower threshold (0.35) catches softer speech common in Bengali,
shorter min_speech (200ms) and longer silence detection (150ms) for natural Bengali pauses.

In [ ]:
# ── Optimized VAD parameters for Bengali speech ──
VAD_THRESHOLD = 0.35       # Lowered from 0.6 — catches softer Bengali speech
VAD_MIN_SPEECH_MS = 200    # Shorter minimum to capture short utterances
VAD_MIN_SILENCE_MS = 150   # Slightly longer silence detection for Bengali pauses
VAD_SPEECH_PAD_MS = 200     # Padding around detected speech


def get_speech_segments_vad(
    audio: np.ndarray,
    sample_rate: int = TARGET_SR,
) -> list:
    """
    Run Silero VAD on audio waveform and return speech segments as
    a list of dicts with 'start' and 'end' keys (in samples).
    """
    # Reset VAD model state for a fresh file
    vad_model.reset_states()

    # Convert numpy to torch tensor
    audio_tensor = torch.from_numpy(audio).float()

    # Get speech timestamps using Silero utility
    speech_timestamps = get_speech_timestamps(
        audio_tensor,
        vad_model,
        threshold=VAD_THRESHOLD,
        sampling_rate=sample_rate,
        min_speech_duration_ms=VAD_MIN_SPEECH_MS,
        min_silence_duration_ms=VAD_MIN_SILENCE_MS,
        speech_pad_ms=VAD_SPEECH_PAD_MS,
        return_seconds=False,  # Return in samples
    )

    return speech_timestamps


print(
    f"VAD segmenter configured: threshold={VAD_THRESHOLD}, "
    f"min_speech={VAD_MIN_SPEECH_MS}ms, min_silence={VAD_MIN_SILENCE_MS}ms, "
    f"speech_pad={VAD_SPEECH_PAD_MS}ms"
)

## 6. VAD Cut & Merge Chunking Strategy

Greedily merge consecutive VAD speech segments into chunks of ≤29 seconds.
Chunk boundaries occur only during silence (between VAD segments), preventing mid-word cuts.

In [ ]:
MAX_CHUNK_DURATION = 29.0  # seconds — fits within Whisper's 30s window
PADDING_SAMPLES = int(0.1 * TARGET_SR)  # 100ms silence padding per side


def merge_segments_into_chunks(
    segments: list,
    audio: np.ndarray,
    sr: int = TARGET_SR,
    max_duration: float = MAX_CHUNK_DURATION,
) -> list:
    """
    Merge VAD speech segments into chunks of at most `max_duration` seconds.
    Each chunk is a contiguous numpy array of audio samples containing only
    speech regions, with boundaries falling during silence.

    Returns:
        List of numpy arrays, each a chunk ready for ASR inference.
    """
    if not segments:
        return []

    max_samples = int(max_duration * sr)
    chunks = []
    current_chunk_segments = []
    current_chunk_len = 0

    for seg in segments:
        seg_start = max(0, seg["start"] - PADDING_SAMPLES)
        seg_end = min(len(audio), seg["end"] + PADDING_SAMPLES)
        seg_len = seg_end - seg_start

        # If a single segment exceeds max duration, split it
        if seg_len > max_samples:
            # Flush current accumulator first
            if current_chunk_segments:
                chunk_audio = np.concatenate(
                    [audio[s:e] for s, e in current_chunk_segments]
                )
                chunks.append(chunk_audio)
                current_chunk_segments = []
                current_chunk_len = 0

            # Split the oversized segment into sub-chunks
            for offset in range(seg_start, seg_end, max_samples):
                sub_end = min(offset + max_samples, seg_end)
                chunks.append(audio[offset:sub_end])
            continue

        # Check if adding this segment exceeds max chunk duration
        if current_chunk_len + seg_len > max_samples:
            # Flush current chunk
            if current_chunk_segments:
                chunk_audio = np.concatenate(
                    [audio[s:e] for s, e in current_chunk_segments]
                )
                chunks.append(chunk_audio)
            current_chunk_segments = [(seg_start, seg_end)]
            current_chunk_len = seg_len
        else:
            current_chunk_segments.append((seg_start, seg_end))
            current_chunk_len += seg_len

    # Flush remaining segments
    if current_chunk_segments:
        chunk_audio = np.concatenate(
            [audio[s:e] for s, e in current_chunk_segments]
        )
        chunks.append(chunk_audio)

    return chunks


print(f"Chunking strategy defined: max_duration={MAX_CHUNK_DURATION}s")

## 7. Batched ASR Inference with Dataset + Anti-Hallucination


In [ ]:
# ── Load mozilla-ai/whisper-large-v3-bn with HuggingFace Transformers ──
# Using transformers directly (not CTranslate2) to ensure task="transcribe"
# is correctly applied and we get Bengali text, not English translation.

from transformers import GenerationConfig

_device = "cuda" if torch.cuda.is_available() else "cpu"
_torch_dtype = torch.float16 if _device == "cuda" else torch.float32

print(f"Loading {WHISPER_MODEL_ID} with HuggingFace Transformers on {_device}...")

hf_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID,
    torch_dtype=_torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa"  # <--- THIS IS THE MAGIC SPEEDUP LINE
).to(_device)

# ── FIX: Update generation config BEFORE creating the pipeline ──
print("Checking generation config for language support...")
if getattr(hf_model.generation_config, "lang_to_id", None) is None:
    print("Missing language mapping. Pulling base config from openai/whisper-medium...")
    base_config = GenerationConfig.from_pretrained("openai/whisper-medium")
    hf_model.generation_config = base_config
    print("Generation config updated successfully!")

# Force extra_special_tokens to be a valid dictionary to bypass the config bug
processor = AutoProcessor.from_pretrained(
    WHISPER_MODEL_ID,
    extra_special_tokens={}
)

# Create ASR pipeline — referenced as `whisper_model` throughout the notebook
whisper_model = hf_pipeline(
    "automatic-speech-recognition",
    model=hf_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=_torch_dtype,
    device=_device,
)

print(f"Model loaded: {WHISPER_MODEL_ID} on {_device} ({_torch_dtype})")

@torch.inference_mode()
def transcribe_chunks(model, chunks: list, batch_size: int = BATCH_SIZE) -> str:
    """
    Transcribe audio chunks using batched HuggingFace ASR pipeline.

    Passes a generator of {"raw": np.ndarray, "sampling_rate": int} dicts
    to the pipeline with `batch_size`, enabling its internal DataLoader to
    batch multiple chunks onto the GPU simultaneously for higher throughput.

    Returns concatenated transcription text.
    """
    if not chunks:
        return ""

    # Filter out empty chunks
    valid_chunks = [c for c in chunks if len(c) > 0]
    if not valid_chunks:
        return ""

    print(
        f"  Batched inference: {len(valid_chunks)} chunks, "
        f"batch_size={batch_size}"
    )
    t_start = time.time()

    # ── Build a generator of audio dicts for the pipeline ──
    # The pipeline accepts any iterable; with batch_size>1 it batches
    # multiple items for parallel GPU forward passes.
    def audio_generator():
        for chunk in valid_chunks:
            yield {"raw": chunk, "sampling_rate": TARGET_SR}

    generate_kwargs = {
        "language": "bengali",
        "task": "transcribe",
        "num_beams": BEAM_SIZE,
        "repetition_penalty": REPETITION_PENALTY,
    }

    all_texts = []
    for i, result in enumerate(
        model(
            audio_generator(),
            batch_size=batch_size,
            generate_kwargs=generate_kwargs,
            # chunk_length_s=30,
        )
    ):
        text = result["text"].strip()
        chunk_dur = len(valid_chunks[i]) / TARGET_SR
        if text:
            all_texts.append(text)
            print(f"    Chunk {i+1}/{len(valid_chunks)} ({chunk_dur:.1f}s): '{text[:70]}...'")
        else:
            logger.warning(f"    Chunk {i+1}/{len(valid_chunks)} ({chunk_dur:.1f}s): empty transcription")

    elapsed = time.time() - t_start
    total_audio = sum(len(c) / TARGET_SR for c in valid_chunks)
    batch_rtf = elapsed / total_audio if total_audio > 0 else 0

    print(
        f"  Batched inference done in {elapsed:.1f}s "
        f"({len(all_texts)}/{len(valid_chunks)} successful, "
        f"batch RTF={batch_rtf:.3f})"
    )
    return " ".join(all_texts)

print(
    f"ASR inference ready: model={WHISPER_MODEL_ID}, beam={BEAM_SIZE}, "
    f"rep_penalty={REPETITION_PENALTY}, batch_size={BATCH_SIZE}, "
    f"task=transcribe, language={LANGUAGE}"
)

## 8. Bengali Number-to-Word Conversion

Maps all digit characters (ASCII 0–9 and Bengali Unicode ০–৯) to their Bengali word forms.

In [ ]:
# ── Digit-to-Bengali-Word Mapping ──
DIGIT_TO_BENGALI_WORD = {
    "0": "শূন্য", "1": "এক", "2": "দুই", "3": "তিন", "4": "চার",
    "5": "পাঁচ", "6": "ছয়", "7": "সাত", "8": "আট", "9": "নয়",
    # Bengali Unicode digits
    "০": "শূন্য", "১": "এক", "২": "দুই", "৩": "তিন", "৪": "চার",
    "৫": "পাঁচ", "৬": "ছয়", "৭": "সাত", "৮": "আট", "৯": "নয়",
}

# Pre-compiled regex for all digit characters (ASCII + Bengali)
_DIGIT_PATTERN = re.compile(r"[0-9০-৯]")


def digits_to_bengali_words(text: str) -> str:
    """
    Replace every digit (ASCII and Bengali Unicode) in the text with
    the corresponding Bengali word, separated by spaces.
    """
    def _replace_digit(match):
        return DIGIT_TO_BENGALI_WORD.get(match.group(0), match.group(0))

    return _DIGIT_PATTERN.sub(_replace_digit, text)


# Quick test
assert digits_to_bengali_words("আমি 3টা বই পড়ি") == "আমি তিনটা বই পড়ি"
assert digits_to_bengali_words("২০২৬") == "দুইশূন্যদুইছয়"
print("Bengali digit-to-word converter ready.")

## 9. Unicode Normalization and Text Cleaning Engine

Every word is passed through `bnunicodenormalizer.Normalizer` to fix Unicode inconsistencies.
A regex-based cleaner strips all punctuation, and digits are converted to Bengali words.

In [ ]:
# ── Initialize Bengali Unicode Normalizer ──
bn_normalizer = Normalizer()

# ── Compiled regex: remove all specified punctuation ──
# Characters: ! " # $ % & ' ( ) * + , - . / : ; < = > ? @ [ \ ] ^ _ ` { | } ~ ।
_PUNCTUATION_PATTERN = re.compile(
    r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~।]'
)

# Collapse multiple whitespace into single space
_MULTI_SPACE = re.compile(r"\s+")


def normalize_transcript(text: str) -> str:
    """
    Full post-processing pipeline for a raw Whisper transcript:
      1. Remove all punctuation characters
      2. Normalize each word with bnunicodenormalizer
      3. Convert digits to Bengali words
      4. Strip whitespace and collapse multi-spaces
    """
    if not text or not text.strip():
        return ""

    # Step 1: Remove punctuation
    text = _PUNCTUATION_PATTERN.sub(" ", text)

    # Step 2: Word-level Bengali Unicode normalization
    words = text.split()
    normalized_words = []
    for word in words:
        normalized = bn_normalizer(word)
        # bn_normalizer returns a dict with 'normalized' key or the string directly
        if isinstance(normalized, dict):
            norm_text = normalized.get("normalized", word)
        elif isinstance(normalized, tuple):
            norm_text = normalized[0] if normalized[0] else word
        else:
            norm_text = str(normalized) if normalized else word

        if norm_text and norm_text.strip():
            normalized_words.append(norm_text.strip())

    text = " ".join(normalized_words)

    # Step 3: Convert digits to Bengali words
    text = digits_to_bengali_words(text)

    # Step 4: Final whitespace cleanup
    text = _MULTI_SPACE.sub(" ", text).strip()

    return text


# Quick validation
print("Normalization engine initialized and tested.")

## 10. Advanced Post-Processing

In [ ]:
class BengaliTextProcessor:
    """
    Advanced Bengali text processor with repetition removal.
    Fixes Whisper hallucinations that produce repeated words/phrases.
    """

    def __init__(self):
        self.whitespace_pattern = re.compile(r"\s+")
        self.zero_width_pattern = re.compile(r"[\u200b\u200c\u200d\ufeff\u200e\u200f]")

    def normalize_unicode(self, text: str) -> str:
        """Normalize Unicode to NFC form."""
        return unicodedata.normalize("NFC", text)

    def remove_zero_width(self, text: str) -> str:
        """Remove zero-width characters."""
        return self.zero_width_pattern.sub("", text)

    def normalize_whitespace(self, text: str) -> str:
        """Normalize whitespace to single spaces."""
        return self.whitespace_pattern.sub(" ", text).strip()

    def remove_repeated_words(self, text: str, max_repeats: int = 2) -> str:
        """
        Remove consecutively repeated words.
        Keeps at most max_repeats consecutive occurrences.
        """
        words = text.split()
        if len(words) <= 1:
            return text

        result = []
        repeat_count = 1

        for i, word in enumerate(words):
            if i == 0:
                result.append(word)
            elif word == words[i - 1]:
                repeat_count += 1
                if repeat_count <= max_repeats:
                    result.append(word)
            else:
                repeat_count = 1
                result.append(word)

        return " ".join(result)

    def remove_repeated_phrases(
        self, text: str, min_phrase_len: int = 2, max_phrase_len: int = 8
    ) -> str:
        """
        Remove repeated n-gram phrases from text.
        Critical for fixing Whisper hallucinations like
        'এটা পোরেবেন সেটে মানারে এটা পোরেবেন সেটে মানারে'
        """
        words = text.split()
        if len(words) < min_phrase_len * 2:
            return text

        for phrase_len in range(max_phrase_len, min_phrase_len - 1, -1):
            i = 0
            new_words = []

            while i < len(words):
                if i + phrase_len * 2 <= len(words):
                    phrase1 = words[i : i + phrase_len]
                    phrase2 = words[i + phrase_len : i + phrase_len * 2]

                    if phrase1 == phrase2:
                        # Count total repetitions and skip them
                        j = i + phrase_len
                        while (
                            j + phrase_len <= len(words)
                            and words[j : j + phrase_len] == phrase1
                        ):
                            j += phrase_len

                        new_words.extend(phrase1)  # Keep one occurrence
                        i = j
                        continue

                new_words.append(words[i])
                i += 1

            words = new_words

        return " ".join(words)

    def remove_excessive_repetition(self, text: str, threshold: float = 0.3) -> str:
        """
        Detect and fix texts with excessive repetition.
        If a single word appears > threshold of the time, aggressively deduplicate.
        """
        words = text.split()
        if len(words) < 10:
            return text

        word_counts = Counter(words)
        most_common_word, most_common_count = word_counts.most_common(1)[0]
        repetition_ratio = most_common_count / len(words)

        if repetition_ratio > threshold:
            seen_bigrams = set()
            result = []
            for i, word in enumerate(words):
                if i == 0:
                    result.append(word)
                else:
                    bigram = (words[i - 1], word)
                    if bigram not in seen_bigrams or len(seen_bigrams) < 100:
                        result.append(word)
                        seen_bigrams.add(bigram)
            return " ".join(result)

        return text

    def process(self, text: str) -> str:
        """Full post-processing pipeline."""
        if not text:
            return ""

        text = self.normalize_unicode(text)
        text = self.remove_zero_width(text)
        text = self.normalize_whitespace(text)
        text = self.remove_repeated_phrases(text)
        text = self.remove_repeated_words(text, max_repeats=2)
        text = self.remove_excessive_repetition(text)
        text = self.normalize_whitespace(text)
        return text


# Initialize processor
text_processor = BengaliTextProcessor()

# Test with known hallucination patterns
test_cases = [
    "বাই বাই বাই বাই বাই বাই বাই বাই বাই বাই বাই",
    "একটো তারা তারি করে একটো তারা তারি করে একটো তারা",
    "এটা পোরেবেন সেটে মানারে এটা পোরেবেন সেটে মানারে এটা পোরেবেন সেটে মানারে",
    "আজ আমরা দীর্ঘ অডিও ট্রান্সক্রিপশন নিয়ে আলোচনা করব",
]

print("Testing repetition removal:")
print("=" * 60)
for test in test_cases:
    result = text_processor.process(test)
    inp = f"'{test[:50]}...'" if len(test) > 50 else f"'{test}'"
    print(f"Input:  {inp}")
    print(f"Output: '{result}'")
    print("-" * 40)

print("BengaliTextProcessor initialized and tested.")

## 11. Processing single file code

In [ ]:
@torch.inference_mode()
def process_single_file(
    filepath: str,
    vad_model_ref,
    whisper_model_ref,
    normalizer_fn,
    post_processor=None,
) -> str:
    """
    Full pipeline for a single audio file:
      1. Load & resample audio → 16 kHz mono
      2. VAD segmentation
      3. Merge segments into ≤29s chunks
      4. Whisper ASR inference (with anti-hallucination)
      5. Normalize & clean transcript
      6. Advanced post-processing (n-gram dedup)
      7. Clear CUDA cache
    """
    fname = os.path.basename(filepath)
    t0 = time.time()

    # ── Stage 1: Load Audio ──
    print(f"[{fname}] Loading audio...")
    audio = load_audio(filepath)
    if len(audio) == 0:
        logger.warning(f"[{fname}] Empty/corrupt audio → returning empty string")
        return ""

    duration = len(audio) / TARGET_SR
    print(f"[{fname}] Loaded: {duration:.1f}s ({len(audio)} samples)")

    # ── Stage 2: VAD Segmentation ──
    print(f"[{fname}] Running VAD segmentation...")
    segments = get_speech_segments_vad(audio, TARGET_SR)
    print(f"[{fname}] VAD segments found: {len(segments)}")

    if len(segments) == 0:
        logger.warning(f"[{fname}] No speech detected → returning empty string")
        clear_cuda_cache()
        return ""

    # Log individual segment durations
    for si, seg in enumerate(segments):
        seg_dur = (seg["end"] - seg["start"]) / TARGET_SR
        # print(f"[{fname}]   Segment {si+1}: {seg_dur:.2f}s")

    # ── Stage 3: Chunk Merging ──
    print(f"[{fname}] Merging segments into chunks (max {MAX_CHUNK_DURATION}s)...")
    chunks = merge_segments_into_chunks(segments, audio, TARGET_SR)
    total_chunk_dur = sum(len(c) / TARGET_SR for c in chunks)
    print(
        f"[{fname}] Chunks: {len(chunks)} | "
        f"Total speech: {total_chunk_dur:.1f}s / {duration:.1f}s"
    )
    # for ci, c in enumerate(chunks):
    #     print(f"[{fname}]   Chunk {ci+1}: {len(c)/TARGET_SR:.2f}s ({len(c)} samples)")

    # ── Stage 4: ASR Inference ──
    print(f"[{fname}] Starting ASR inference on {len(chunks)} chunks...")
    t_asr = time.time()
    raw_transcript = transcribe_chunks(whisper_model_ref, chunks)
    asr_elapsed = time.time() - t_asr
    print(f"[{fname}] ASR completed in {asr_elapsed:.1f}s")
    print(f"[{fname}] Raw transcript ({len(raw_transcript.split())} words): {raw_transcript[:120]}...")

    # ── Stage 5: Normalization ──
    # print(f"[{fname}] Normalizing transcript...")
    # final_transcript = normalizer_fn(raw_transcript)
    # print(f"[{fname}] Normalized ({len(final_transcript.split())} words): {final_transcript}...")

    final_transcript = raw_transcript
    # ── Stage 6: Advanced Post-Processing (n-gram dedup) ──
    if post_processor is not None:
        original_len = len(final_transcript.split())
        final_transcript = post_processor.process(final_transcript)
        clean_len = len(final_transcript.split())
        removed = original_len - clean_len
        if original_len > 0 and removed > 0:
            print(
                f"[{fname}] Dedup: removed {removed} repeated words "
                f"({100*removed/original_len:.1f}%) → {clean_len} words remaining"
            )
        else:
            print(f"[{fname}] Dedup: no repetitions removed")

    print(f"[{fname}] Final transcript ({len(final_transcript.split())} words): {final_transcript}...")

    # ── Stage 7: Cleanup ──
    clear_cuda_cache()

    elapsed = time.time() - t0
    rtf = elapsed / duration if duration > 0 else 0
    print(f"[{fname}] DONE in {elapsed:.1f}s | RTF={rtf:.3f}")
    print(f"  ✓ {fname}: {len(final_transcript.split())} words | {elapsed:.1f}s | RTF={rtf:.3f}")

    return final_transcript


print("Pipeline orchestration function ready.")

# 12. Main Inference

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION — Dual T4 parallel inference via threading
#
# Architecture:
#   1. Load model_gpu0 on cuda:0  \  sequential in main thread (no file-lock race)
#   2. Load model_gpu1 on cuda:1  /
#   3. Thread-0 runs inference on cuda:0 for files[0:half]
#   4. Thread-1 runs inference on cuda:1 for files[half:]
#      GIL is released during CUDA ops → both GPUs run truly in parallel
#   5. Merge results in original file order
# ══════════════════════════════════════════════════════════════════════════════
import os, gc, glob, time, threading
import numpy as np
import pandas as pd
import torch
import librosa
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, GenerationConfig
from transformers import pipeline as hf_pipeline

# ── Discover test files ──
audio_extensions = ["*.mp3", "*.wav", "*.flac", "*.ogg", "*.m4a", "*.webm"]
test_files = []
for ext in audio_extensions:
    test_files.extend(glob.glob(os.path.join(TEST_AUDIO_DIR, ext)))
if not test_files:
    test_files = [f for f in glob.glob(os.path.join(TEST_AUDIO_DIR, "*"))
                  if os.path.isfile(f)]
test_files = sorted(test_files)
print(f"Found {len(test_files)} test audio files")

pipeline_start = time.time()

# ══════════════════════════════════════════════════════════════════════════════
if USE_DUAL_GPU and len(test_files) >= 2:

    # ── Step 1: Load both models SEQUENTIALLY (no file-lock contention) ──
    print("\nLoading models sequentially (avoids safetensors file-lock races)...")

    def load_whisper_on_device(device_str):
        """Load Whisper onto a specific device string e.g. 'cuda:0'"""
        _dtype = torch.float16
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            WHISPER_MODEL_ID,
            torch_dtype=_dtype,
            low_cpu_mem_usage=True,
            use_safetensors=True,
            attn_implementation="sdpa",
        ).to(device_str)
        if getattr(model.generation_config, "lang_to_id", None) is None:
            base_cfg = GenerationConfig.from_pretrained("openai/whisper-medium")
            model.generation_config = base_cfg
        proc = AutoProcessor.from_pretrained(WHISPER_MODEL_ID, extra_special_tokens={})
        pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=model,
            tokenizer=proc.tokenizer,
            feature_extractor=proc.feature_extractor,
            torch_dtype=_dtype,
            device=device_str,
        )
        return pipe

    print("  Loading model on cuda:0 ...")
    pipe0 = load_whisper_on_device("cuda:0")
    print("  cuda:0 ready ✓")

    print("  Loading model on cuda:1 ...")
    pipe1 = load_whisper_on_device("cuda:1")
    print("  cuda:1 ready ✓")

    print("Both models loaded. Starting parallel inference via threads.\n")

    # ── Step 2: Thread worker — uses a pre-loaded pipeline ──
    def thread_worker(pipe, file_paths, thread_id, out_list):
        """
        Runs on one thread. Uses the pre-loaded pipeline (already on its GPU).
        GIL is released during CUDA forward passes so both threads run in parallel.
        """
        generate_kwargs = {
            "language": "bengali",
            "task": "transcribe",
            "num_beams": BEAM_SIZE,
            "repetition_penalty": REPETITION_PENALTY,
        }

        for idx, fpath in enumerate(file_paths):
            fname = os.path.basename(fpath)
            print(f"  [Thread {thread_id}] [{idx+1}/{len(file_paths)}] {fname}")
            t0 = time.time()

            # Load audio
            audio = load_audio(fpath)
            if len(audio) == 0:
                out_list.append({"filename": fname, "transcript": ""})
                continue

            # VAD — note: vad_model is shared but reset_states + inference
            # is protected by a lock since it holds internal state
            with vad_lock:
                segments = get_speech_segments_vad(audio, TARGET_SR)

            if not segments:
                out_list.append({"filename": fname, "transcript": ""})
                continue

            chunks = merge_segments_into_chunks(segments, audio, TARGET_SR)
            valid  = [c for c in chunks if len(c) > 0]

            def audio_gen():
                for c in valid:
                    yield {"raw": c, "sampling_rate": TARGET_SR}

            texts = []
            with torch.inference_mode():
                for result in pipe(
                    audio_gen(),
                    batch_size=BATCH_SIZE,
                    generate_kwargs=generate_kwargs,
                    # chunk_length_s=30,
                ):
                    t = result["text"].strip()
                    if t:
                        texts.append(t)

            raw   = " ".join(texts)
            final = text_processor.process(raw)
            dur   = len(audio) / TARGET_SR
            elapsed = time.time() - t0
            print(f"  [Thread {thread_id}] ✓ {fname} | {elapsed:.1f}s | "
                  f"{len(final.split())} words | RTF={elapsed/dur:.3f}")
            # print(final)
            out_list.append({"filename": fname, "transcript": final})

    # VAD model is stateful — protect reset_states + inference with a lock
    vad_lock = threading.Lock()

    half   = len(test_files) // 2
    split0 = test_files[:half]
    split1 = test_files[half:]

    # Thread-safe result lists (each thread appends only to its own list)
    results0, results1 = [], []

    t0_thread = threading.Thread(target=thread_worker, args=(pipe0, split0, 0, results0))
    t1_thread = threading.Thread(target=thread_worker, args=(pipe1, split1, 1, results1))

    print(f"🚀 Launching threads: Thread-0 → {len(split0)} files on cuda:0 | "
          f"Thread-1 → {len(split1)} files on cuda:1")
    t0_thread.start()
    t1_thread.start()
    t0_thread.join()
    t1_thread.join()
    print("\n✅ Both threads complete.")

    # Merge results in original file order
    all_results_map = {item["filename"]: item["transcript"]
                       for item in results0 + results1}
    results = [
        {"filename": os.path.splitext(os.path.basename(fp))[0],
         "transcript": all_results_map.get(os.path.basename(fp), "")}
        for fp in test_files
    ]
    print(f"Merged {len(results)} results in original file order.")

    # Free GPU memory
    del pipe0, pipe1
    torch.cuda.empty_cache()
    gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
else:
    # Single-GPU fallback — original pipeline, unchanged
    print("\n⚡ Single-GPU mode (original pipeline)")
    results = []
    with torch.inference_mode():
        for idx, fpath in enumerate(test_files):
            print(f"{'='*60}")
            print(f"Processing [{idx+1}/{len(test_files)}]: {os.path.basename(fpath)}")
            transcription = process_single_file(
                filepath=fpath,
                vad_model_ref=vad_model,
                whisper_model_ref=whisper_model,
                normalizer_fn=normalize_transcript,
                post_processor=text_processor,
            )
            results.append({"filename": os.path.splitext(os.path.basename(fpath))[0],
                            "transcript": transcription})
            if torch.cuda.is_available() and (idx + 1) % 3 == 0:
                torch.cuda.empty_cache()
                gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
# STATS & SUBMISSION
# ══════════════════════════════════════════════════════════════════════════════
pipeline_elapsed = time.time() - pipeline_start
total_audio_duration = sum(
    librosa.get_duration(path=fp) for fp in test_files if os.path.isfile(fp)
)
avg_rtf = pipeline_elapsed / total_audio_duration if total_audio_duration > 0 else 0

print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE")
print(f"Total files:           {len(test_files)}")
print(f"Total wall-clock time: {pipeline_elapsed:.1f}s")
print(f"Total audio duration:  {total_audio_duration:.1f}s")
print(f"Average RTF:           {avg_rtf:.4f}")

submission_df = pd.DataFrame(results)
if os.path.exists(SAMPLE_SUBMISSION_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    target_cols = list(sample_sub.columns)
    if len(target_cols) >= 2:
        submission_df.columns = target_cols[:2]

# submission_df.to_csv(SUBMISSION_PATH, index=False)
# print(f"Submission saved: {SUBMISSION_PATH} | shape: {submission_df.shape}")
submission_df.head(10)


## 13. Saving

In [ ]:
import pandas as pd
import re

# ── ASCII Digit-to-Bengali Word Mapping ──
ASCII_TO_BENGALI = {
    "0": "শূন্য", "1": "এক", "2": "দুই", "3": "তিন", "4": "চার",
    "5": "পাঁচ", "6": "ছয়", "7": "সাত", "8": "আট", "9": "নয়",
}

# Regex Explanation:
# (?<=[\u0980-\u09FF])\d : Digit preceded by a Bengali character
# \d(?=[\u0980-\u09FF]) : Digit followed by a Bengali character
_CONTEXTUAL_DIGIT_PATTERN = re.compile(r"(?<=[\u0980-\u09FF])[0-9]|[0-9](?=[\u0980-\u09FF])")
_PUNCTUATION_PATTERN = re.compile(
    r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~।]'
)
_MULTI_SPACE = re.compile(r"\s+")

def normalize_contextual_digits(text: str) -> str:
    """
    Replace ASCII digits with Bengali words ONLY if they are 
    attached to Bengali text (e.g., '3টা' -> 'তিনটা').
    Standalone numbers (e.g., '40') remain unchanged.
    """
    def _replace_digit(match):
        return ASCII_TO_BENGALI.get(match.group(0), match.group(0))

    text = _PUNCTUATION_PATTERN.sub(" ", text)
    text = _MULTI_SPACE.sub(" ", text).strip()
    return _CONTEXTUAL_DIGIT_PATTERN.sub(_replace_digit, text)

# ── Verification ──

# Case 1: Attached to Bengali (Should convert)
res1 = normalize_contextual_digits("আমি 3টা বই পড়ি")
print(f"Case 1: {res1}") # আমি তিনটা বই পড়ি

# Case 2: Standalone Number (Should stay)
res2 = normalize_contextual_digits("আমার বয়স 20")
print(f"Case 2: {res2}") # আমার বয়স 20

# Case 3: Mixed (Should stay)
res3 = normalize_contextual_digits("২০২৬")
print(f"Case 3: {res3}") # ২০২৬

# Assertions
assert res1 == "আমি তিনটা বই পড়ি"
assert res2 == "আমার বয়স 20"
assert res3 == "২০২৬"


# 3. Apply the function to the specific column containing the audio text
# Replace 'transcript' with your actual column name if it's different
text_column_name = 'transcript'

# The .apply() method runs your function on every single row in that column
submission_df[text_column_name] = submission_df[text_column_name].apply(normalize_contextual_digits)

# 4. Save the updated dataframe to a new CSV
output_csv_path = "submission.csv"
submission_df.to_csv(output_csv_path, index=False)

print(f"✓ Function applied successfully! Saved to {output_csv_path}")